# Tree Search Comparison: PIP vs TKF

This notebook analyzes the performance and accuracy of **Poisson Indel Process (PIP)** and **Thorne-Kishino-Felsenstein (TKF)** models based on the results in `results/summary.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import numpy as np
import statsmodels.api as sm

# Configuration
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load data
df = pd.read_csv('results/summary.tsv', sep='\t')

# Note: The 'gap_strategy' column seems to contain the model name (PIP/TKF92)
model_col = 'gap_strategy'

# Ensure numeric types for metrics
metrics = ['robinson_foulds', 'kuhner_felsenstein', 'start_robinson_foulds', 'start_kuhner_felsenstein', 'runtime_seconds', 'log_likelihood', 'alignment_length', 'gap_percentage', 'species']
for col in metrics:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Loaded {len(df)} results.")
df.head()

## 1. Topological Accuracy (Robinson-Foulds Distance)

The Robinson-Foulds (RF) distance measures the symmetric difference between the inferred tree and the true tree. **Lower is better.**

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = pd.melt(df, id_vars=['gap_strategy', 'species'], value_vars=['robinson_foulds', 'start_robinson_foulds'], var_name='tree_type', value_name='rf')
plot_df['model'] = plot_df.apply(lambda x: 'NJ (Start)' if x['tree_type'] == 'start_robinson_foulds' else x['gap_strategy'], axis=1)
sns.boxplot(data=plot_df, x='model', y='rf', hue='model', palette='Set2', legend=False)
sns.stripplot(data=plot_df, x='model', y='rf', color='black', alpha=0.3)
plt.title('Topological Accuracy Comparison (Robinson-Foulds)')
plt.ylabel('Robinson-Foulds Distance')
plt.xlabel('Model Type')
plt.show()

## 3. Computational Complexity and Runtime Analysis

Comparing runtimes across model types and tree sizes.

In [ ]:
# Filter data for efficiency analysis (exclude non-positive runtimes)
efficiency_df = df[df['runtime_seconds'] > 0].copy()

# Stratified Runtime Plot: MSA Length on X-axis, Hue by Model, Faceted by Tree Size
g = sns.FacetGrid(efficiency_df, col="species", hue=model_col, height=5, aspect=1.2, sharey=False)
g.map(sns.scatterplot, "alignment_length", "runtime_seconds", s=100)
g.map(sns.regplot, "alignment_length", "runtime_seconds", scatter=False, order=1)
g.add_legend()
g.set_axis_labels("Alignment Length (bp)", "Runtime (seconds)")
plt.subplots_adjust(top=0.85)
g.fig.suptitle('Runtime vs Alignment Length: Stratified by Tree Size (Species)')
plt.show()

## 4. Branch Length Accuracy (Kuhner-Felsenstein)

The Kuhner-Felsenstein distance accounts for both topology and branch length estimation accuracy.

In [ ]:
if 'kuhner_felsenstein' in df.columns:
    plt.figure(figsize=(12, 6))
    plot_df_kf = pd.melt(df, id_vars=['gap_strategy', 'species'], value_vars=['kuhner_felsenstein', 'start_kuhner_felsenstein'], var_name='tree_type', value_name='kf')
    plot_df_kf['model'] = plot_df_kf.apply(lambda x: 'NJ (Start)' if x['tree_type'] == 'start_kuhner_felsenstein' else x['gap_strategy'], axis=1)
    sns.boxplot(data=plot_df_kf, x='model', y='kf', hue='model', palette='Set1', legend=False)
    plt.title('Branch Length Estimation Comparison (Kuhner-Felsenstein)')
    plt.ylabel('Kuhner-Felsenstein Distance')
    plt.xlabel('Model Type')
    plt.show()

## 5. Stratified Accuracy Analysis

We compare **Robinson-Foulds (RF)** and **Kuhner-Felsenstein (KF)** distances across different **tree sizes** (number of species) and **MSA simulation tools**.

In [ ]:
def plot_stratified(metric, title_prefix):
    if metric not in df.columns: return
    
    start_metric = f'start_{metric}'
    if start_metric in df.columns:
        # Reshape data to include start tree as a model category
        melted = pd.melt(df, id_vars=['gap_strategy', 'species', 'msa_sim_tool'], 
                         value_vars=[metric, start_metric], 
                         var_name='temp_type', value_name='val')
        melted['model_cat'] = melted.apply(lambda x: 'NJ (Start)' if x['temp_type'] == start_metric else x['gap_strategy'], axis=1)
        
        # 1. Stratify by Tree Size (Species)
        plt.figure(figsize=(14, 7))
        sns.boxplot(data=melted, x='species', y='val', hue='model_cat', palette='Set2')
        plt.title(f'{title_prefix} Stratified by Tree Size (Species)')
        plt.ylabel(metric)
        plt.xlabel('Number of Species')
        plt.show()

        # 2. Stratify by MSA Simulation Tool
        if 'msa_sim_tool' in df.columns:
            plt.figure(figsize=(14, 7))
            sns.boxplot(data=melted, x='msa_sim_tool', y='val', hue='model_cat', palette='Set3')
            plt.title(f'{title_prefix} Stratified by MSA Simulation Tool')
            plt.ylabel(metric)
            plt.xlabel('Simulation Tool')
            plt.show()
    else:
        # Fallback to original behavior if start metric is missing
        plt.figure(figsize=(14, 7))
        sns.boxplot(data=df, x='species', y=metric, hue=model_col, palette='Set2')
        plt.title(f'{title_prefix} Stratified by Tree Size (Species)')
        plt.ylabel(metric)
        plt.xlabel('Number of Species')
        plt.show()
        if 'msa_sim_tool' in df.columns:
            plt.figure(figsize=(14, 7))
            sns.boxplot(data=df, x='msa_sim_tool', y=metric, hue=model_col, palette='Set3')
            plt.title(f'{title_prefix} Stratified by MSA Simulation Tool')
            plt.ylabel(metric)
            plt.xlabel('Simulation Tool')
            plt.show()

plot_stratified('robinson_foulds', 'Topological Accuracy (RF)')
plot_stratified('kuhner_felsenstein', 'Branch Length Accuracy (KF)')